# BARISimulation 학습 노트북

팀원 저장소의 `barisimulation train`을 Jupyter에서 실행할 수 있게 옮긴 노트북입니다. 학습 방식은 원래 코드(`bari_sim/workflows/training.py`)와 같습니다. 모든 로봇이 공유하는 선형 정책의 가중치를 교차 엔트로피 방법(CEM)으로 찾습니다. 여기에 아래 기능을 더했습니다.

- **진행 상황 표시:** 후보마다 점수, 걸린 시간, 남은 예상 시간을 출력합니다.
- **폭주 후보 처리:** 물리 계산이 폭주(NaN)한 후보는 로봇을 처음 자세로 되돌리지 않고 "실패" 점수를 줍니다. 저장소 코드는 수정하지 않습니다.
- **중단 후 이어서 학습:** 세대마다 checkpoint와 그때까지의 최고 모델을 저장합니다.
- **결과 확인:** 학습 곡선, 평가, GIF 녹화, MuJoCo 뷰어를 제공합니다.

**준비 (Anaconda Prompt, 처음 한 번)**

```bash
conda activate barisimulation
python -m pip install notebook ipykernel matplotlib pillow
cd <BARISimulation 폴더>
jupyter notebook
```

이 노트북 파일은 `BARISimulation` 폴더 안에 두고 여세요. 커널이 `barisimulation` 환경인지 확인하세요.


In [ ]:
import json, math, sys, time
from dataclasses import replace
from pathlib import Path

import numpy as np
import mujoco

import bari_sim
from bari_sim.policies import LinearPolicy, PolicyMetadata
from bari_sim.simulation import SceneRequest, Simulation
from bari_sim.tasks import parse_robot_grid, task_definition

# 저장소 루트 찾기(이 노트북이 BARISimulation 폴더 안에 있다고 가정)
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'pyproject.toml').exists() and (candidate / 'bari_sim').is_dir():
        PROJECT_ROOT = candidate
        break
print('Python', sys.version.split()[0], '| MuJoCo', mujoco.__version__)
print('bari_sim:', Path(bari_sim.__file__).parent)
print('프로젝트 폴더:', PROJECT_ROOT)


## 1. 학습 설정

`QUICK_TEST = True`로 두면 1세대 × 후보 4개 × 30초만 돌려서 한 후보에 걸리는 시간을 먼저 잴 수 있습니다. 확인한 뒤 `False`로 바꿔 본 학습을 돌리세요.

In [ ]:
QUICK_TEST = True          # 먼저 True로 시간 측정 → 이후 False

ROBOTS = '2x5'             # 줄 x 열 (2x5, 4x5, 6x5 ...)
TASK = 'gap'               # 'collision-avoidance', 'gap', 'step'
DIFFICULTY = 1             # 1~5 (gap: 0.10/0.15/0.20/0.25/0.30 m)
GENERATIONS = 20
POPULATION = 32
ELITE_FRACTION = 0.25
DURATION_S = 120.0         # 후보 하나당 시뮬레이션 시간(초)
SEED = 7
INITIAL_SCALE = 0.75       # 원래 코드와 같은 초기 탐색 폭
MIN_SCALE = 0.05
UNSTABLE_SCORE = -1.0e6    # 물리 폭주 후보의 점수
RESUME = True              # 같은 설정의 checkpoint가 있으면 이어서 학습

if QUICK_TEST:
    GENERATIONS, POPULATION, DURATION_S = 1, 4, 30.0

grid = parse_robot_grid(ROBOTS)
task = task_definition(TASK, DIFFICULTY)
RUN_NAME = f"{TASK}-d{DIFFICULTY}-{grid.rows}x{grid.columns}" + ('-quick' if QUICK_TEST else '')
OUTPUT = PROJECT_ROOT / 'models' / f'{RUN_NAME}.json'
CHECKPOINT = PROJECT_ROOT / 'models' / f'{RUN_NAME}.checkpoint.json'
print(f'과제 {task.name.value}, 난이도 {DIFFICULTY} ({task.value_name}={task.value} {task.unit}), 로봇 {grid.count}대')
print(f'{GENERATIONS}세대 x 후보 {POPULATION}개 x {DURATION_S:.0f}초 = 시뮬레이션 {GENERATIONS*POPULATION}회')
print('모델 저장:', OUTPUT)


## 2. 폭주 감지 시뮬레이션

MuJoCo는 계산이 폭주하면 기본적으로 로봇을 처음 자세로 되돌리고 계속 진행합니다(autoreset). 그러면 그 후보가 엉터리 실행으로 채점됩니다. 아래 클래스는 이 기능을 끄고, 제어 한 주기(0.5초)가 끝날 때마다 폭주 여부를 검사합니다. 저장소 파일은 건드리지 않습니다.

In [ ]:
class SimulationUnstable(RuntimeError):
    pass

_BAD_WARNINGS = [int(mujoco.mjtWarning.mjWARN_BADQACC),
                 int(mujoco.mjtWarning.mjWARN_BADQPOS),
                 int(mujoco.mjtWarning.mjWARN_BADQVEL)]

class GuardedSimulation(Simulation):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model.opt.disableflags |= int(mujoco.mjtDisableBit.mjDSBL_AUTORESET)

    def is_unstable(self):
        if any(self.data.warning[k].number > 0 for k in _BAD_WARNINGS):
            return True
        return not (np.isfinite(self.data.qpos).all() and np.isfinite(self.data.qvel).all())

    def reset(self):
        observations = super().reset()
        for k in _BAD_WARNINGS:
            try:
                self.data.warning[k].number = 0
            except Exception:
                pass
        return observations

    def step(self, *args, **kwargs):
        result = super().step(*args, **kwargs)
        if self.is_unstable():
            raise SimulationUnstable(f'MuJoCo 계산 폭주 (t={self.time_s:.2f}s)')
        return result


def rollout(simulation, policy, duration_s, **run_kwargs):
    """(score, TaskResult 또는 None, 상태 문자열)을 돌려준다."""
    simulation.reset()
    try:
        result = simulation.run(lambda _rid, obs: policy.act(obs), duration_s, **run_kwargs)
    except Exception as error:   # 저장소에 패치를 적용했다면 그쪽 SimulationUnstable도 여기로 온다
        if isinstance(error, SimulationUnstable) or type(error).__name__ == 'SimulationUnstable' \
                or simulation.is_unstable():
            return UNSTABLE_SCORE, None, f'unstable: {error}'
        raise
    return float(result.metrics['score']), result, 'ok'


simulation = GuardedSimulation(SceneRequest(grid=grid, environment=task.environment, task=task))
print('시뮬레이션 준비 완료: 로봇', simulation.robot_count, '대, 물리 step', simulation.timestep_s, 's')


## 3. 학습 (CEM)

원래 코드와 같은 순서로 학습합니다.

1. 첫 평균은 "정지" 정책으로 두고, 탐색 폭은 0.75로 시작합니다.
2. 매 세대 후보를 정규분포에서 뽑습니다. 0번 후보는 항상 현재 평균입니다.
3. 점수 상위 25% 후보의 평균과 표준편차(최소 0.05)로 다음 세대의 분포를 정합니다.

세대마다 `models/<RUN_NAME>.checkpoint.json`과 그때까지의 최고 모델을 저장합니다. 커널을 멈췄다가 이 셀을 다시 실행하면 이어서 학습합니다.

In [ ]:
metadata = PolicyMetadata(task=task.name.value, difficulty=task.difficulty,
                          robots=str(grid), seed=SEED)
signature = dict(robots=str(grid), task=TASK, difficulty=DIFFICULTY, population=POPULATION,
                 elite_fraction=ELITE_FRACTION, duration_s=DURATION_S, seed=SEED,
                 initial_scale=INITIAL_SCALE, min_scale=MIN_SCALE)

rng = np.random.default_rng(SEED)
mean = LinearPolicy.idle(metadata).parameters()
scale = np.full(LinearPolicy.PARAMETER_COUNT, INITIAL_SCALE)
best_parameters, best_score = mean.copy(), -math.inf
history, start_generation = [], 0

if RESUME and CHECKPOINT.exists():
    saved = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
    if saved['signature'] == signature:
        mean = np.array(saved['mean']); scale = np.array(saved['scale'])
        best_parameters = np.array(saved['best_parameters']); best_score = saved['best_score']
        history = saved['history']; start_generation = saved['completed_generations']
        rng.bit_generator.state = saved['rng_state']
        print(f'checkpoint에서 이어서 학습: {start_generation}세대 완료, 최고 점수 {best_score:.4f}')
    else:
        print('설정이 달라 checkpoint를 무시하고 새로 시작합니다.')

def save_best():
    meta = replace(metadata, training_score=float(best_score))
    LinearPolicy.from_parameters(best_parameters, meta).save(OUTPUT)

elite_count = max(1, round(POPULATION * ELITE_FRACTION))
episode_times = []
train_start = time.perf_counter()

for generation in range(start_generation, GENERATIONS):
    population = rng.normal(mean, scale, size=(POPULATION, LinearPolicy.PARAMETER_COUNT))
    population[0] = mean
    scores = np.empty(POPULATION)
    statuses = []
    for index, parameters in enumerate(population):
        t0 = time.perf_counter()
        policy = LinearPolicy.from_parameters(parameters, metadata)
        score, result, status = rollout(simulation, policy, DURATION_S)
        scores[index] = score; statuses.append(status)
        episode_times.append(time.perf_counter() - t0)
        remaining = (GENERATIONS - generation) * POPULATION - (index + 1)
        eta_min = remaining * float(np.mean(episode_times)) / 60
        extra = ''
        if result is not None:
            m = result.metrics
            extra = (f" 성공={result.success} 성공비율={m.get('successful_fraction', float('nan')):.2f}"
                     f" 충돌={m.get('collision_count')} 뒤집힘={m.get('flipped_immobile_robot_count')}")
        print(f'[세대 {generation+1}/{GENERATIONS} 후보 {index+1:2d}/{POPULATION}] '
              f'점수={score:.4f} ({episode_times[-1]:.0f}s){extra}'
              f"{'' if status == 'ok' else '  <- ' + status}  | 남은 시간 약 {eta_min:.0f}분", flush=True)

    ranked = np.argsort(scores)[::-1]
    elites = population[ranked[:elite_count]]
    mean = elites.mean(axis=0)
    scale = np.maximum(elites.std(axis=0), MIN_SCALE)
    if scores[ranked[0]] > best_score:
        best_score = float(scores[ranked[0]]); best_parameters = population[ranked[0]].copy()
    valid = scores[scores > UNSTABLE_SCORE]
    history.append(dict(generation=generation + 1, best=float(scores[ranked[0]]),
                        median=float(np.median(valid)) if len(valid) else None,
                        elite_mean=float(scores[ranked[:elite_count]].mean()),
                        unstable=int(sum(s != 'ok' for s in statuses)),
                        overall_best=float(best_score)))
    save_best()
    CHECKPOINT.write_text(json.dumps(dict(
        signature=signature, completed_generations=generation + 1,
        mean=mean.tolist(), scale=scale.tolist(), best_parameters=best_parameters.tolist(),
        best_score=best_score, history=history, rng_state=rng.bit_generator.state),
        indent=1), encoding='utf-8')
    print(f"== 세대 {generation+1} 요약: 최고 {history[-1]['best']:.4f}, 중앙값 {history[-1]['median']}, "
          f"폭주 {history[-1]['unstable']}개, 전체 최고 {best_score:.4f} -> {OUTPUT.name} 저장", flush=True)

print(f'학습 종료. 전체 최고 점수 {best_score:.4f}, 경과 {(time.perf_counter()-train_start)/60:.1f}분')
print('모델:', OUTPUT)


## 4. 학습 곡선

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows 한글 글꼴
    plt.rcParams['axes.unicode_minus'] = False
    g = [h['generation'] for h in history]
    plt.figure(figsize=(7, 4))
    plt.plot(g, [h['best'] for h in history], 'o-', label='세대 최고')
    plt.plot(g, [h['elite_mean'] for h in history], 's-', label='상위 후보 평균')
    plt.plot(g, [h['median'] if h['median'] is not None else float('nan') for h in history], '^-', label='중앙값(폭주 제외)')
    plt.xlabel('세대'); plt.ylabel('score'); plt.grid(alpha=.3); plt.legend()
    plt.show()
except ImportError:
    print('matplotlib이 없습니다: python -m pip install matplotlib')
for h in history:
    print(h)


## 5. 평가

저장된 최고 모델로 여러 번 실행해 성공률과 평균 점수를 봅니다. 초기 배치가 고정이라 매번 결과가 같을 수 있습니다. 그래도 로봇 수나 난이도를 바꿔 일반화 정도를 확인할 수 있습니다.

In [ ]:
EVAL_MODEL = OUTPUT
EVAL_ROBOTS = ROBOTS
EVAL_DIFFICULTY = DIFFICULTY
EVAL_EPISODES = 3
EVAL_DURATION_S = 120.0

eval_policy = LinearPolicy.load(Path(EVAL_MODEL))
eval_grid = parse_robot_grid(EVAL_ROBOTS)
eval_task = task_definition(TASK, EVAL_DIFFICULTY)
eval_sim = GuardedSimulation(SceneRequest(grid=eval_grid, environment=eval_task.environment, task=eval_task))
eval_results = []
for episode in range(EVAL_EPISODES):
    score, result, status = rollout(eval_sim, eval_policy, EVAL_DURATION_S)
    eval_results.append((score, result, status))
    if result is None:
        print(f'에피소드 {episode+1}: {status}')
    else:
        m = result.metrics
        print(f"에피소드 {episode+1}: 성공={result.success} 점수={score:.4f} "
              f"성공비율={m.get('successful_fraction')} 시간={result.elapsed_time_s:.1f}s "
              f"충돌={m.get('collision_count')} 뒤집힘={m.get('flipped_immobile_robot_count')}")
ok = [r for _, r, _ in eval_results if r is not None]
if ok:
    print(f'성공률 {sum(r.success for r in ok)/len(eval_results):.2f}, '
          f'평균 점수 {np.mean([s for s, r, _ in eval_results if r is not None]):.4f}')
    print(json.dumps(ok[-1].as_dict(), indent=2, ensure_ascii=False, default=str))


## 6. GIF로 녹화 (선택)

`pillow`가 필요합니다. 로봇 무리의 중심을 따라가며 초당 `GIF_FPS`장으로 녹화합니다.

In [ ]:
RECORD_GIF = False
GIF_SECONDS = 60.0
GIF_FPS = 5
GIF_PATH = PROJECT_ROOT / 'models' / f'{RUN_NAME}.gif'

if RECORD_GIF:
    from PIL import Image
    from IPython.display import Image as IPImage, display
    rec_sim = GuardedSimulation(SceneRequest(grid=eval_grid, environment=eval_task.environment, task=eval_task))
    renderer = mujoco.Renderer(rec_sim.model, 360, 640)
    camera = mujoco.MjvCamera()
    camera.type = mujoco.mjtCamera.mjCAMERA_FREE
    camera.azimuth, camera.elevation = 135, -28
    camera.distance = max(1.4, 0.35 * eval_grid.rows + 1.2)
    frames, state = [], {'next': 0.0}
    def grab(sim):
        if sim.time_s + 1e-9 >= state['next']:
            camera.lookat[:] = sim.data.xpos[list(sim.root_body_ids)].mean(axis=0)
            renderer.update_scene(sim.data, camera=camera)
            frames.append(Image.fromarray(renderer.render()))
            state['next'] += 1.0 / GIF_FPS
        return None
    score, result, status = rollout(rec_sim, eval_policy, GIF_SECONDS, frame_callback=grab)
    renderer.close()
    print('상태:', status, '| 점수:', score, '| 프레임:', len(frames))
    if frames:
        frames[0].save(GIF_PATH, save_all=True, append_images=frames[1:],
                       duration=int(1000 / GIF_FPS), loop=0)
        print('저장:', GIF_PATH)
        display(IPImage(filename=str(GIF_PATH)))


## 7. MuJoCo 뷰어로 보기 (선택)

별도 창이 열립니다. 창을 닫거나 시간이 끝나면 셀이 끝납니다. 명령창에서 `barisimulation infer --robots 2x5 --task gap --model models\<파일>.json --viewer`로 실행해도 같은 화면을 볼 수 있습니다.

In [ ]:
SHOW_VIEWER = False
VIEWER_SECONDS = 120.0

if SHOW_VIEWER:
    from bari_sim.workflows import run_inference
    view_sim = GuardedSimulation(SceneRequest(grid=eval_grid, environment=eval_task.environment, task=eval_task))
    try:
        view_result = run_inference(view_sim, eval_policy, duration_s=VIEWER_SECONDS, viewer_enabled=True)
        print(json.dumps(view_result.as_dict(), indent=2, ensure_ascii=False, default=str))
    except SimulationUnstable as error:
        print('뷰어 실행 중 폭주:', error)
